In [1]:
import os
import sys
from pathlib import Path
print(os.getcwd())

ROOT_DIR = Path(os.getcwd()).parent.parent
print(ROOT_DIR)
sys.path.insert(0, str(ROOT_DIR))

d:\Project\CAPSTONE_PROJECT\videodeepsearch\test\test_notebook
d:\Project\CAPSTONE_PROJECT\videodeepsearch


In [3]:
import sys, pkgutil

print("Python:", sys.executable)
print("Paths:", sys.path[:3])
print("Agent exists:", "agent" in [p.name for p in pkgutil.iter_modules([r'D:\Project\CAPSTONE_PROJECT\videodeepsearch'])])


Python: d:\Project\CAPSTONE_PROJECT\.venv\Scripts\python.exe
Paths: ['C:\\ProgramData\\anaconda3\\python312.zip', 'C:\\ProgramData\\anaconda3\\DLLs', 'C:\\ProgramData\\anaconda3\\Lib']
Agent exists: True


In [4]:
from agent.tools.clients import *
from agent.tools.schema.artifact import ImageObjectInterface, SegmentObjectInterface, VideoInterface
from agent.tools.type import *

ModuleNotFoundError: No module named 'agent.tools'

In [ ]:
# CONSTANT
MILVUS_HOST = "localhost"
MILVUS_PORT = 19530
MILVUS_URI = f"http://{MILVUS_HOST}:{MILVUS_PORT}"

IMAGE_COLLECTION_NAME = "image_milvus"
IMAGE_VISUAL_PARAM = {
    "type_config": "dense",
    "dimension": 512,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Image embeddings for video frames",
    "index_params": {"M": 64, "efConstruction": 100},
}
IMAGE_CAPTION_DENSE_PARAM = {
    "type_config": "dense",
    "dimension": 768,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Text embeddings for image captions",
    "index_params": {"M": 64, "efConstruction": 100},
}
IMAGE_CAPTION_SPARSE_PARAM = {
    "type_config": "sparse",
    "dimension": 1000000,  # ignored at runtime
    "metric_type": "BM25",
    "index_type": "SPARSE_INVERTED_INDEX",
    "description": "BM25 sparse index for image captions",
    "index_params": {"inverted_index_algo": "DAAT_MAXSCORE"},
}
IMAGE_VISUAL_FIELD = "visual_embedding_field"
IMAGE_CAPTION_FIELD = "caption_embedding_field"
IMAGE_SPARSE_FIELD = "caption_sparse_embedding_field"


SEGMENT_CAPTION_COLLECTION_NAME = "segment_milvus"
SEGMENT_CAPTION_DENSE_PARAM = {
    "type_config": "dense",
    "dimension": 768,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Text embeddings for segment captions",
    "index_params": {"M": 64, "efConstruction": 100},
}
SEGMENT_CAPTION_SPARSE_PARAM = {
    "type_config": "sparse",
    "dimension": 1000000,  # ignored at runtime
    "metric_type": "BM25",
    "index_type": "SPARSE_INVERTED_INDEX",
    "description": "BM25 sparse index for segment captions",
    "index_params": {"inverted_index_algo": "DAAT_MAXSCORE"},
}

SEGMENT_DENSE_FIELD = "caption_embedding_field"
SEGMENT_SPARSE_FIELD = "caption_sparse_embedding_field"



In [ ]:
image_milvus_client = ImageMilvusClient(
    uri=MILVUS_URI,
    collection_name=IMAGE_COLLECTION_NAME,
    visual_param=IMAGE_VISUAL_PARAM,
    caption_param=IMAGE_CAPTION_DENSE_PARAM,
    sparse_param=IMAGE_CAPTION_SPARSE_PARAM,
    visual_ann_field=IMAGE_VISUAL_FIELD,
    caption_ann_field=IMAGE_CAPTION_FIELD,
    sparse_field=IMAGE_SPARSE_FIELD,
)

segment_caption_client = SegmentCaptionImageMilvusClient(
    uri=MILVUS_URI,
    collection_name=SEGMENT_CAPTION_COLLECTION_NAME,
    dense_param=SEGMENT_CAPTION_DENSE_PARAM,
    sparse_param=SEGMENT_CAPTION_SPARSE_PARAM,
    dense_field=SEGMENT_DENSE_FIELD,
    sparse_field=SEGMENT_SPARSE_FIELD,
)

await image_milvus_client.connect()
await segment_caption_client.connect()

In [ ]:
# await image_milvus_client.close()
# await segment_caption_client.close()

In [ ]:
image_emebedding_settings = ImageEmbeddingSettings(
    model_name='open_clip',
    device='cuda',
    batch_size=32
)
text_emebedding_settings = TextEmbeddingSettings(
    model_name='mmbert',
    device='cuda',
    batch_size=32
)

img_txt_client = ImageEmbeddingClient(base_url='http://localhost:8003')
txt_client = TextEmbeddingClient(base_url='http://localhost:8005')


external_client = ExternalEncodeClient(img_text_client=img_txt_client, img_text_settings=image_emebedding_settings, txt_settings=text_emebedding_settings, txt_client=txt_client)

await external_client.connect()

In [ ]:
MINIO_HOST = "localhost"       # use localhost when running locally
MINIO_PORT = 9000
MINIO_USER = "minioadmin"
MINIO_PASSWORD = "minioadmin"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
MINIO_SECURE = False  
MINIO_ENDPOINT = f"{MINIO_HOST}:{MINIO_PORT}"

storage_client = StorageClient(
    host=MINIO_HOST,
    port=MINIO_PORT,#type:ignore
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=MINIO_SECURE,
)

storage_client._ensure_bucket('anonymous')

In [ ]:
from sqlalchemy import text

POSTGRE_USER = "prefect"
POSTGRE_PASSWORD = "prefect"
POSTGRE_HOST = "localhost"       # use localhost for local setup
POSTGRE_PORT = 5432
POSTGRE_DB = "prefect"

POSTGRE_DATABASE_URL = (
    f"postgresql+asyncpg://{POSTGRE_USER}:{POSTGRE_PASSWORD}"
    f"@{POSTGRE_HOST}:{POSTGRE_PORT}/{POSTGRE_DB}"
)
postgres_client = PostgresClient(
    database_url=POSTGRE_DATABASE_URL
)
async with postgres_client.get_session() as session:
    result = await session.execute(text("SELECT version();"))
    version = result.scalar_one()
    print(f"🗄️ PostgreSQL connection successful.\n   Version: {version}")

In [ ]:

from llama_index.llms.google_genai import GoogleGenAI
from dotenv import load_dotenv
load_dotenv()


llm = GoogleGenAI(
    model='gemini-2.5-flash',
        
)

tool_factory = ToolFactory(
    image_milvus_client=image_milvus_client,
    segment_milvus_client=segment_caption_client,
    postgres_client=postgres_client,
    minio_client=storage_client,
    external_client=external_client,
    llm_as_tools=llm
)

In [ ]:
tools = tool_factory.get_all_tools()

In [ ]:
LIST_VIDEO_IDS = ['video1_111', 'video2_222', 'video3_333']
USER_ID = 'anonymous'

In [ ]:
# testing search tools

VISUAL_QUERY = "traffic police officer in beige uniform and red cap sitting at a roadside table with a man in black Adidas jacket, reviewing papers; traffic cones and another masked person nearby"

from typing import Callable
search_images: Callable = tools['get_images_from_visual_query']
search_images

In [ ]:
result = await search_images(
    visual_query=VISUAL_QUERY,
    top_k=10,
    list_video_id=LIST_VIDEO_IDS,
    user_id=USER_ID
)

In [ ]:
import json
json.loads(result.text)

In [ ]:
search_images_caption: Callable = tools['get_images_from_caption_query']
search_images_caption

In [ ]:
CAPTION_QUERY = "Cảnh sát giao thông Việt Nam mặc đồng phục màu be, đội mũ kêpi viền đỏ, ngồi bên bàn ven đường cùng một người đàn ông mặc áo khoác Adidas đen, cùng xem giấy tờ; gần đó có người đeo khẩu trang và các cọc tiêu giao thông màu cam."
result_caption = await search_images_caption(
    caption_query=CAPTION_QUERY,
    top_k_each_request=50,
    top_k_final=30,
    weights=None,
    list_video_id=LIST_VIDEO_IDS,
    user_id=USER_ID
)

In [ ]:
CAPTION_QUERY = "Cảnh sát giao thông Việt Nam mặc đồng phục màu be, đội mũ kêpi viền đỏ, ngồi bên bàn ven đường cùng một người đàn ông mặc áo khoác Adidas đen, cùng xem giấy tờ; gần đó có người đeo khẩu trang và các cọc tiêu giao thông màu cam."
result_caption = await search_images_caption(
    caption_query=CAPTION_QUERY,
    top_k_each_request=50,
    top_k_final=30,
    weights=(0.7, 0.3),
    list_video_id=LIST_VIDEO_IDS,
    user_id=USER_ID
)

In [ ]:
get_segment_from_q_func: Callable = tools['get_segments_from_event_query']

result_segment = await get_segment_from_q_func(
    event_query=CAPTION_QUERY,
    top_k_each_request=50,
    top_k_final=30,
    weights=(0.7, 0.3),
    list_video_id=LIST_VIDEO_IDS,
    user_id=USER_ID
)


In [ ]:
multimodal_searching: Callable = tools['get_images_from_multimodal_query']

In [ ]:
result_mul = await multimodal_searching(
    visual_query=VISUAL_QUERY,
    caption_query=CAPTION_QUERY,
    top_k_each_request=50,
    top_k_final=30,
    weights=(0.45, 0.45, 0.05),
    list_video_id=LIST_VIDEO_IDS,
    user_id=USER_ID
)

In [ ]:
import json
result_mulll = json.loads(result_mul.text)
result_mulll

In [ ]:
find_similar: Callable = tools['find_similar_images_from_image']

image_ref = ImageObjectInterface.model_validate(result_mulll['results']['video1_111'][0])

result_ref = await find_similar(
    reference_image=image_ref,
    top_k=10,
    list_video_id=LIST_VIDEO_IDS,
    user_id=USER_ID
)


In [ ]:
json.loads(result_ref.text)

In [ ]:
res_seg = json.loads(result_segment.text)
res_seg

In [ ]:
sample_segment_objects = []
for video_id, segments in res_seg['results'].items():
    for segment in segments:
            
        segment_obj = {
            'related_video_id': video_id,
            **segment
        }
        segobj = SegmentObjectInterface.model_validate(segment_obj)
        sample_segment_objects.append(segobj)
    

len(sample_segment_objects)

In [ ]:
sample_image_interface = []
for video_id, images in json.loads(result_caption.text)['results'].items():
    for img in images:
        img_obj = {
            'related_video_id': video_id,
            ** img
        }
        ioj = ImageObjectInterface.model_validate(img_obj)
        sample_image_interface.append(ioj)

# Testing the utils tools

In [ ]:
get_vid_from_seg: Callable = tools['get_video_from_segment']
video_retrieve = await get_vid_from_seg(sample_segment_objects[0])
video_retrieve

In [ ]:
get_vid_from_img = tools['get_video_from_image']
video_retrieve = await get_vid_from_img(sample_image_interface[0])
video_retrieve

In [ ]:
get_asr = tools['get_asr_from_video']
asr = await get_asr(video_retrieve)
asr

In [ ]:
get_all_seg = tools['get_all_segment_info_from_video_interface']
segments = await get_all_seg(video_retrieve)
segments

In [ ]:

sample_segment_objects[10]

In [ ]:
get_segs: Callable= tools['get_segments']
forward = await get_segs(
    current_segment=sample_segment_objects[10],
    hop=3,
    include_within_range=True,
    forward_or_backward='forward'
)

In [ ]:
forward

In [ ]:
# get_img: Callable = tools['get_images']
# images = await get_img(
#     image=sample_image_interface[9],
#     hop=3,
#     include_within_range=True,
#     forward_or_backward='backward'
# )

In [ ]:
images

In [ ]:
sample_image_interface[10]

In [ ]:
'00:18:50.200'

In [ ]:
extract_f = tools['extract_frame_time']
extract_obj_interface = await extract_f(
    video_interface=video_retrieve,
    timestamp='00:18:50.200',
    agent_bucket='test-agent',
    agent_object_folder='test1'
)

In [ ]:
extract_obj_interface

In [ ]:
extract_f2: Callable = tools['extract_frames_by_time_window']
extract_f2_obj = await extract_f2(
    video_interface=video_retrieve,
    start_time='00:18:50.200',
    end_time='00:20:00.000',
    fps_sample_rate=60,
    agent_bucket='test-agent',
    agent_object_folder='test1'
)

In [ ]:
json.loads(extract_f2_obj.text)

In [ ]:
read_img = tools['read_image']
img_b = read_img(sample_image_interface[0])

In [ ]:
# read_segment = tools['read_segment']
# segment_b = await read_segment(sample_segment_objects[0])

In [ ]:
get_asr_fi = tools['get_related_asr_from_image']

asrrrr = await get_asr_fi(sample_image_interface[0], window_seconds=30.0)

In [ ]:
get_asr_fs = tools['get_related_asr_from_segment']
asr_seg =  await get_asr_fs(sample_segment_objects[0], window_seconds=30.0)

In [ ]:
asr_seg

In [ ]:
sample_segment_objects[0]

In [ ]:
ftt = tools['frame_to_timecode']
timecode=  ftt(
    frame_index=10000,
    fps=25
)

In [ ]:
timecode

In [ ]:
llm_enhance_tools = tools['enhance_visual_query']

result_enhanced_query = await llm_enhance_tools(
    raw_query="Hãy tìm cho tôi khoảnh khắc của một con chó đang chạy nhảy trên cỏ",
    variants=['Hãy nâng cấp không gian', 'làm rõ hơn về ngữ cảnh', 'Thêm một số tình tiết liên quan']
)

In [ ]:
result_enhanced_query